# Pet Breed Classification: Custom CNN vs Pre-trained CNN

**Dataset:** Oxford-IIIT Pet Dataset  
**Task:** Multi-class image classification  
**Main aim:** Compare a custom CNN trained from scratch with a pre-trained CNN using transfer learning.

This notebook includes:
1. Dataset loading and exploration  
2. Preprocessing and augmentation  
3. Custom CNN baseline  
4. Pre-trained CNN using MobileNetV2  
5. Evaluation using accuracy, precision, recall, F1-score, classification report, and confusion matrix  
6. Final comparison table  
7. Prediction function for new images  


In [ ]:
# ============================================================
# 1. Install / import libraries
# ============================================================

import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
import tensorflow_datasets as tfds

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support
)

print("TensorFlow version:", tf.__version__)

In [ ]:
# ============================================================
# 2. Load Oxford-IIIT Pet Dataset
# ============================================================

# as_supervised=True gives data as (image, label)
dataset, info = tfds.load(
    "oxford_iiit_pet",
    as_supervised=True,
    with_info=True
)

train_full = dataset["train"]
test_raw = dataset["test"]

class_names = info.features["label"].names
NUM_CLASSES = info.features["label"].num_classes

print("Dataset name:", info.name)
print("Number of classes:", NUM_CLASSES)
print("Training examples:", info.splits["train"].num_examples)
print("Test examples:", info.splits["test"].num_examples)
print("First 10 class names:", class_names[:10])

## Dataset exploration

This section checks the number of images, classes, image examples, and class distribution. This is important because class imbalance or visually similar classes can affect model performance.

In [ ]:
# ============================================================
# 3. Visualize sample images
# ============================================================

plt.figure(figsize=(12, 12))

for i, (image, label) in enumerate(train_full.take(9)):
    plt.subplot(3, 3, i + 1)
    plt.imshow(image)
    plt.title(class_names[int(label)])
    plt.axis("off")

plt.suptitle("Sample Images from Oxford-IIIT Pet Dataset", fontsize=16)
plt.show()

In [ ]:
# ============================================================
# 4. Check image shapes
# ============================================================

for image, label in train_full.take(5):
    print("Image shape:", image.shape, "| Label:", int(label), "| Class:", class_names[int(label)])

In [ ]:
# ============================================================
# 5. Class distribution
# ============================================================

def get_label_counts(ds, class_names):
    labels = []
    for _, label in tfds.as_numpy(ds):
        labels.append(int(label))
    counts = pd.Series(labels).value_counts().sort_index()
    df = pd.DataFrame({
        "class_id": counts.index,
        "class_name": [class_names[i] for i in counts.index],
        "count": counts.values
    })
    return df

train_counts = get_label_counts(train_full, class_names)
test_counts = get_label_counts(test_raw, class_names)

display(train_counts.head())

plt.figure(figsize=(16, 6))
plt.bar(train_counts["class_name"], train_counts["count"])
plt.xticks(rotation=90)
plt.xlabel("Class")
plt.ylabel("Number of Images")
plt.title("Training Set Class Distribution")
plt.tight_layout()
plt.show()

## Preprocessing

Both models use the same image size and the same train/validation/test split.  
This makes the comparison fair.

- Images are resized to 224 × 224.
- Training data is shuffled.
- A validation set is created from the training set.
- Datasets are batched and prefetched for faster training.


In [ ]:
# ============================================================
# 6. Preprocessing settings
# ============================================================

IMG_SIZE = 224
BATCH_SIZE = 32
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE

TRAIN_SIZE = info.splits["train"].num_examples
VAL_SIZE = int(0.2 * TRAIN_SIZE)

print("Training size before validation split:", TRAIN_SIZE)
print("Validation size:", VAL_SIZE)
print("Final training size:", TRAIN_SIZE - VAL_SIZE)

In [ ]:
# ============================================================
# 7. Resize images
# ============================================================

def resize_image(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32)
    return image, label

# Shuffle once, then split into train and validation sets
train_shuffled = train_full.shuffle(
    buffer_size=TRAIN_SIZE,
    seed=SEED,
    reshuffle_each_iteration=False
)

val_raw = train_shuffled.take(VAL_SIZE)
train_raw = train_shuffled.skip(VAL_SIZE)

train_ds = (
    train_raw
    .map(resize_image, num_parallel_calls=AUTOTUNE)
    .shuffle(1000, seed=SEED)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_ds = (
    val_raw
    .map(resize_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

test_ds = (
    test_raw
    .map(resize_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print("Datasets prepared successfully.")

In [ ]:
# ============================================================
# 8. Data augmentation layer
# ============================================================

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
], name="data_augmentation")

# Model 1: Custom CNN

The custom CNN is built from scratch. It starts with random weights and learns all features directly from this dataset.

This model is used as the **baseline** for comparison.

In [ ]:
# ============================================================
# 9. Build Custom CNN
# ============================================================

def build_custom_cnn(num_classes):
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

    x = data_augmentation(inputs)
    x = tf.keras.layers.Rescaling(1./255)(x)

    x = tf.keras.layers.Conv2D(32, (3, 3), activation="relu", padding="same")(x)
    x = tf.keras.layers.MaxPooling2D()(x)

    x = tf.keras.layers.Conv2D(64, (3, 3), activation="relu", padding="same")(x)
    x = tf.keras.layers.MaxPooling2D()(x)

    x = tf.keras.layers.Conv2D(128, (3, 3), activation="relu", padding="same")(x)
    x = tf.keras.layers.MaxPooling2D()(x)

    x = tf.keras.layers.Conv2D(256, (3, 3), activation="relu", padding="same")(x)
    x = tf.keras.layers.MaxPooling2D()(x)

    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(256, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.4)(x)

    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

    model = tf.keras.Model(inputs, outputs, name="Custom_CNN")
    return model

custom_cnn = build_custom_cnn(NUM_CLASSES)

custom_cnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

custom_cnn.summary()

In [ ]:
# ============================================================
# 10. Train Custom CNN
# ============================================================

callbacks_custom = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-6
    )
]

start_time = time.time()

history_custom = custom_cnn.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks_custom
)

custom_training_time = time.time() - start_time
print(f"Custom CNN training time: {custom_training_time:.2f} seconds")

In [ ]:
# ============================================================
# 11. Plot Custom CNN training curves
# ============================================================

def plot_history(history, title):
    hist = pd.DataFrame(history.history)

    plt.figure(figsize=(8, 5))
    plt.plot(hist["accuracy"], label="Training Accuracy")
    plt.plot(hist["val_accuracy"], label="Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(title + " - Accuracy")
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(hist["loss"], label="Training Loss")
    plt.plot(hist["val_loss"], label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title + " - Loss")
    plt.legend()
    plt.grid(True)
    plt.show()

plot_history(history_custom, "Custom CNN")

# Model 2: Pre-trained CNN using Transfer Learning

The pre-trained CNN uses MobileNetV2 trained on ImageNet.  
Instead of learning all visual features from scratch, it reuses features learned from a large dataset.

We replace the original classification head with a new output layer for the 37 pet classes.

In [ ]:
# ============================================================
# 12. Build Pre-trained CNN using MobileNetV2
# ============================================================

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet"
)

# Freeze base model first
base_model.trainable = False

def build_pretrained_model(base_model, num_classes):
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

    x = data_augmentation(inputs)
    x = preprocess_input(x)

    x = base_model(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)

    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

    model = tf.keras.Model(inputs, outputs, name="MobileNetV2_Transfer_Learning")
    return model

pretrained_cnn = build_pretrained_model(base_model, NUM_CLASSES)

pretrained_cnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

pretrained_cnn.summary()

In [ ]:
# ============================================================
# 13. Train Pre-trained CNN with frozen base
# ============================================================

callbacks_pretrained = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-6
    )
]

start_time = time.time()

history_pretrained = pretrained_cnn.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks_pretrained
)

pretrained_training_time_1 = time.time() - start_time
print(f"Pre-trained CNN frozen-base training time: {pretrained_training_time_1:.2f} seconds")

In [ ]:
# ============================================================
# 14. Fine-tune top layers of MobileNetV2
# ============================================================

# Unfreeze the base model
base_model.trainable = True

# Freeze most layers, fine-tune only the last layers
fine_tune_at = len(base_model.layers) - 30

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# Keep BatchNorm layers frozen for more stable fine-tuning
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

pretrained_cnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

start_time = time.time()

history_finetune = pretrained_cnn.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8,
    callbacks=callbacks_pretrained
)

pretrained_training_time_2 = time.time() - start_time
pretrained_total_training_time = pretrained_training_time_1 + pretrained_training_time_2

print(f"Pre-trained CNN fine-tuning time: {pretrained_training_time_2:.2f} seconds")
print(f"Pre-trained CNN total training time: {pretrained_total_training_time:.2f} seconds")

In [ ]:
# ============================================================
# 15. Plot Pre-trained CNN training curves
# ============================================================

plot_history(history_pretrained, "Pre-trained CNN - Frozen Base")
plot_history(history_finetune, "Pre-trained CNN - Fine-tuning")

# Evaluation

We evaluate both models using:

- Accuracy
- Precision
- Recall
- F1-score
- Classification report
- Confusion matrix

This is stronger than using accuracy alone.

In [ ]:
# ============================================================
# 16. Helper function for predictions and metrics
# ============================================================

def get_predictions(model, dataset):
    y_true = []
    y_pred = []

    for images, labels in dataset:
        preds = model.predict(images, verbose=0)
        preds = np.argmax(preds, axis=1)

        y_true.extend(labels.numpy())
        y_pred.extend(preds)

    return np.array(y_true), np.array(y_pred)


def evaluate_model(model, dataset, model_name, training_time):
    y_true, y_pred = get_predictions(model, dataset)

    accuracy = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )

    print("=" * 80)
    print(model_name)
    print("=" * 80)
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-score : {f1:.4f}")
    print(f"Training time: {training_time:.2f} seconds")

    print("\nClassification Report:")
    print(classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        zero_division=0
    ))

    return {
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "Training Time (seconds)": training_time,
        "y_true": y_true,
        "y_pred": y_pred
    }

In [ ]:
# ============================================================
# 17. Evaluate both models
# ============================================================

custom_results = evaluate_model(
    custom_cnn,
    test_ds,
    "Custom CNN",
    custom_training_time
)

pretrained_results = evaluate_model(
    pretrained_cnn,
    test_ds,
    "Pre-trained CNN: MobileNetV2",
    pretrained_total_training_time
)

In [ ]:
# ============================================================
# 18. Model comparison table
# ============================================================

comparison_df = pd.DataFrame([
    {
        "Model": custom_results["Model"],
        "Accuracy": custom_results["Accuracy"],
        "Precision": custom_results["Precision"],
        "Recall": custom_results["Recall"],
        "F1-score": custom_results["F1-score"],
        "Training Time (seconds)": custom_results["Training Time (seconds)"]
    },
    {
        "Model": pretrained_results["Model"],
        "Accuracy": pretrained_results["Accuracy"],
        "Precision": pretrained_results["Precision"],
        "Recall": pretrained_results["Recall"],
        "F1-score": pretrained_results["F1-score"],
        "Training Time (seconds)": pretrained_results["Training Time (seconds)"]
    }
])

display(comparison_df)

comparison_df.set_index("Model")[["Accuracy", "Precision", "Recall", "F1-score"]].plot(
    kind="bar",
    figsize=(10, 6)
)
plt.title("Custom CNN vs Pre-trained CNN Performance")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.grid(axis="y")
plt.show()

In [ ]:
# ============================================================
# 19. Confusion matrices
# ============================================================

def plot_confusion_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(18, 14))
    sns.heatmap(
        cm,
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
        cbar=True
    )
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.title(title)
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

plot_confusion_matrix(
    custom_results["y_true"],
    custom_results["y_pred"],
    "Confusion Matrix - Custom CNN"
)

plot_confusion_matrix(
    pretrained_results["y_true"],
    pretrained_results["y_pred"],
    "Confusion Matrix - Pre-trained CNN"
)

In [ ]:
# ============================================================
# 20. Show example predictions
# ============================================================

def show_predictions(model, dataset, num_images=9):
    plt.figure(figsize=(12, 12))

    images_batch, labels_batch = next(iter(dataset))
    predictions = model.predict(images_batch, verbose=0)
    predicted_labels = np.argmax(predictions, axis=1)

    for i in range(num_images):
        plt.subplot(3, 3, i + 1)

        image = images_batch[i].numpy().astype("uint8")
        true_label = int(labels_batch[i])
        pred_label = int(predicted_labels[i])

        plt.imshow(image)
        color = "green" if true_label == pred_label else "red"
        plt.title(
            f"True: {class_names[true_label]}\nPred: {class_names[pred_label]}",
            color=color,
            fontsize=9
        )
        plt.axis("off")

    plt.tight_layout()
    plt.show()

show_predictions(pretrained_cnn, test_ds, num_images=9)

# Save models and predict new images

Use this section to save the trained models and test a new image.

In [ ]:
# ============================================================
# 21. Save trained models
# ============================================================

custom_cnn.save("custom_cnn_pet_classifier.keras")
pretrained_cnn.save("pretrained_mobilenetv2_pet_classifier.keras")

print("Models saved successfully.")

In [ ]:
# ============================================================
# 22. Function to predict a new image
# ============================================================

from tensorflow.keras.preprocessing import image as keras_image

def predict_pet_breed(model, image_path, class_names, img_size=224):
    img = keras_image.load_img(image_path, target_size=(img_size, img_size))
    img_array = keras_image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)

    predictions = model.predict(img_array, verbose=0)
    predicted_index = np.argmax(predictions[0])
    confidence = predictions[0][predicted_index]

    print("Predicted class:", class_names[predicted_index])
    print("Confidence:", round(float(confidence) * 100, 2), "%")

    plt.imshow(img)
    plt.title(f"Prediction: {class_names[predicted_index]} ({confidence:.2f})")
    plt.axis("off")
    plt.show()

# Example:
# predict_pet_breed(pretrained_cnn, "/content/my_pet_image.jpg", class_names)

# Final conclusion template

Use this wording in your notebook/report after you run the models and fill in your real results:

> The custom CNN provided a useful baseline because it learned features directly from the Oxford-IIIT Pet Dataset. However, the pre-trained MobileNetV2 model achieved better performance because it reused visual features learned from ImageNet. The comparison shows that transfer learning is more suitable for this hackathon task because it trains faster, generalizes better, and performs well even with limited training time. Future improvements could include trying EfficientNetB0 or ResNet50, tuning more hyperparameters, using stronger augmentation, and addressing class-level errors shown in the confusion matrix.
